In [1]:
import pandas as pd

In [2]:
df=pd.read_csv("../data/raw_checked.csv")
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34


### Data Cleaning

In [11]:
# Remove missing customer id
df = df.dropna(subset=['CustomerID'])
# convert customer id to string 
df['CustomerID'] = df['CustomerID'].astype(str)
# Remove negative Quantities
df = df[df['Quantity'] > 0]
# Remove zero or negative prices
df = df[df['UnitPrice'] > 0]

df["InvoiceDate"]=pd.to_datetime(df["InvoiceDate"])

#### Feature Engeenering

In [12]:
snapshot_date = df['InvoiceDate'].max()
# Recency (R) => Days since Last Purchase
rececny=df.groupby('CustomerID')['InvoiceDate'].max().reset_index()
rececny["Recency"]=(snapshot_date-rececny["InvoiceDate"]).dt.days

# Frequency (F) => Number of transactions
frequency = df.groupby('CustomerID')['InvoiceNo'].nunique().reset_index()
frequency.columns = ['CustomerID', 'Frequency']

# Monetary (M) => Total Money Spend
monetary=df.groupby('CustomerID')["TotalPrice"].sum().reset_index()
monetary.columns=["CustomerID","Monetary"]

# MErge RFM
rfm = rececny.merge(frequency, on='CustomerID')
rfm = rfm.merge(monetary, on='CustomerID')

rfm.head()


,CustomerID,InvoiceDate,Recency,Frequency,Monetary
0,12346.0,2011-01-18 10:01:00,325,1,77183.60
1,12347.0,2011-12-07 15:52:00,1,7,4310.00
2,12348.0,2011-09-25 13:13:00,74,4,1797.24
3,12349.0,2011-11-21 09:51:00,18,1,1757.55
4,12350.0,2011-02-02 16:01:00,309,1,334.40


In [15]:
rfm=rfm[["CustomerID","Recency","Frequency","Monetary"]]

In [17]:
rfm.describe()

,Recency,Frequency,Monetary
count,4338.000000,4338.000000,4338.000000
mean,91.536422,4.272015,2054.266460
std,100.014169,7.697998,8989.230441
min,0.000000,1.000000,3.750000
25%,17.000000,1.000000,307.415000
50%,50.000000,2.000000,674.485000
75%,141.000000,5.000000,1661.740000
max,373.000000,209.000000,280206.020000


In [18]:
# Handle Outliers
rfm = rfm[(rfm['Monetary'] < rfm['Monetary'].quantile(0.99))]
rfm = rfm[(rfm['Frequency'] < rfm['Frequency'].quantile(0.99))]

In [21]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

rfm_scaled = scaler.fit_transform(rfm[['Recency', 'Frequency', 'Monetary']])

In [22]:
rfm.to_csv("../data/customer_features.csv", index=False)